<a href="https://colab.research.google.com/github/heetaamin/ml-assignment2/blob/main/final/Trees_Notebook2_Modelling_ipynbipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# load data and configure CV
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import TomekLinks
from imblearn.combine import SMOTETomek

DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 2/networkTraffic_tree_semiraw.csv'
df_tree = pd.read_csv(DATA_PATH)

X = df_tree.drop(columns=['attack_cat'])
y = df_tree['attack_cat']
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("X shape:", X.shape, "y shape:", y.shape)

Mounted at /content/drive
X shape: (162745, 38) y shape: (162745,)


In [2]:
# bucket proto, one-hot encode

nominal_cols = ['proto', 'state', 'service']

fold_data_full = []
for fold_num, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    proto_counts = X_train['proto'].value_counts()
    threshold = 0.01 * len(X_train)
    keep_protos = proto_counts[proto_counts >= threshold].index.tolist()
    X_train['proto'] = X_train['proto'].where(X_train['proto'].isin(keep_protos), 'other')
    X_test['proto'] = X_test['proto'].where(X_test['proto'].isin(keep_protos), 'other')

    X_train_enc = pd.get_dummies(X_train, columns=nominal_cols)
    X_test_enc = pd.get_dummies(X_test, columns=nominal_cols)
    X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

    fold_data_full.append((X_train_enc, X_test_enc, y_train, y_test))
    print(f"Fold {fold_num+1}: prepared. X_train: {X_train_enc.shape}, kept protos: {keep_protos}")

Fold 1: prepared. X_train: (130196, 61), kept protos: ['tcp', 'udp', 'unas']
Fold 2: prepared. X_train: (130196, 61), kept protos: ['tcp', 'udp', 'unas']
Fold 3: prepared. X_train: (130196, 62), kept protos: ['tcp', 'udp', 'unas']
Fold 4: prepared. X_train: (130196, 62), kept protos: ['tcp', 'udp', 'unas']
Fold 5: prepared. X_train: (130196, 61), kept protos: ['tcp', 'udp', 'unas']


column count can differ slightly per fold - service isn't bucketed, so a rare value missing from a fold's training split just means one fewer dummy column, handled by the reindex below



# Impurity measure

In [3]:
# baseline model - gini vs entropy
f1_gini = [f1_score(y_test, DecisionTreeClassifier(criterion='gini', random_state=42).fit(X_train, y_train).predict(X_test), average='macro')
           for X_train, X_test, y_train, y_test in fold_data_full]
print(f"Baseline (gini, unrestricted): {np.mean(f1_gini):.4f} (± {np.std(f1_gini):.4f})")

f1_entropy = [f1_score(y_test, DecisionTreeClassifier(criterion='entropy', random_state=42).fit(X_train, y_train).predict(X_test), average='macro')
              for X_train, X_test, y_train, y_test in fold_data_full]
print(f"Baseline (entropy, unrestricted): {np.mean(f1_entropy):.4f} (± {np.std(f1_entropy):.4f})")

Baseline (gini, unrestricted): 0.5337 (± 0.0099)
Baseline (entropy, unrestricted): 0.5393 (± 0.0092)


# Depth and leaf-size grid search

In [4]:
# max_depth / min_samples_leaf grid search
# entropy fixed, single representative fold
X_train_f, X_test_f, y_train_f, y_test_f = fold_data_full[0]
depths = [5, 10, 15, 20, 25, None]
leaf_sizes = [1, 5, 20, 50]

grid_results = []
for depth in depths:
    for leaf in leaf_sizes:
        model = DecisionTreeClassifier(criterion='entropy', max_depth=depth,
                                        min_samples_leaf=leaf, random_state=42)
        model.fit(X_train_f, y_train_f)
        macro_f1 = f1_score(y_test_f, model.predict(X_test_f), average='macro')
        grid_results.append({'max_depth': depth, 'min_samples_leaf': leaf, 'macro_f1': macro_f1})
        print(f"max_depth={str(depth):5s} min_samples_leaf={leaf:3d}  macro-F1={macro_f1:.4f}")

best = max(grid_results, key=lambda r: r["macro_f1"])
print("\nBest so far:", best)

max_depth=5     min_samples_leaf=  1  macro-F1=0.3040
max_depth=5     min_samples_leaf=  5  macro-F1=0.3040
max_depth=5     min_samples_leaf= 20  macro-F1=0.3039
max_depth=5     min_samples_leaf= 50  macro-F1=0.3039
max_depth=10    min_samples_leaf=  1  macro-F1=0.5425
max_depth=10    min_samples_leaf=  5  macro-F1=0.5379
max_depth=10    min_samples_leaf= 20  macro-F1=0.5292
max_depth=10    min_samples_leaf= 50  macro-F1=0.5218
max_depth=15    min_samples_leaf=  1  macro-F1=0.5641
max_depth=15    min_samples_leaf=  5  macro-F1=0.5570
max_depth=15    min_samples_leaf= 20  macro-F1=0.5513
max_depth=15    min_samples_leaf= 50  macro-F1=0.5378
max_depth=20    min_samples_leaf=  1  macro-F1=0.5597
max_depth=20    min_samples_leaf=  5  macro-F1=0.5567
max_depth=20    min_samples_leaf= 20  macro-F1=0.5527
max_depth=20    min_samples_leaf= 50  macro-F1=0.5447
max_depth=25    min_samples_leaf=  1  macro-F1=0.5472
max_depth=25    min_samples_leaf=  5  macro-F1=0.5532
max_depth=25    min_samples_

max_depth=15, min_samples_leaf=5 carried forward - close to best on this fold and less likely to be single-fold noise than min_samples_leaf=1. Re-validated in Section 7 after resampling is finalised, same as k was for kNN.

# Resampling strategy

In [5]:
# resampling strategy comparison
# entropy, max_depth=15, min_samples_leaf=5 - carried forward
# n_jobs=-1 on TomekLinks - default neighbour search is slow in approx 50 dimensions

from scipy import stats
f1_flat, f1_smote, f1_ros, f1_tomek, f1_smote_tomek = [], [], [], [], []

for fold_num, (X_train, X_test, y_train, y_test) in enumerate(fold_data_full):
    target_flat = {cls: max(count, 5000) for cls, count in y_train.value_counts().items()}
    Xs, ys = SMOTE(sampling_strategy=target_flat, random_state=42, k_neighbors=5).fit_resample(X_train, y_train)
    f1_flat.append(f1_score(y_test, DecisionTreeClassifier(criterion='entropy', max_depth=15, min_samples_leaf=5, random_state=42).fit(Xs, ys).predict(X_test), average='macro'))
    print(f"  fold {fold_num+1}: flat done")

    target_capped = {cls: min(count * 5, y_train.value_counts().max()) for cls, count in y_train.value_counts().items()}
    Xs, ys = SMOTE(sampling_strategy=target_capped, random_state=42, k_neighbors=5).fit_resample(X_train, y_train)
    f1_smote.append(f1_score(y_test, DecisionTreeClassifier(criterion='entropy', max_depth=15, min_samples_leaf=5, random_state=42).fit(Xs, ys).predict(X_test), average='macro'))
    print(f"  fold {fold_num+1}: smote done")

    Xr, yr = RandomOverSampler(sampling_strategy=target_capped, random_state=42).fit_resample(X_train, y_train)
    f1_ros.append(f1_score(y_test, DecisionTreeClassifier(criterion='entropy', max_depth=15, min_samples_leaf=5, random_state=42).fit(Xr, yr).predict(X_test), average='macro'))
    print(f"  fold {fold_num+1}: ros done")

    Xt, yt = TomekLinks(n_jobs=-1).fit_resample(X_train, y_train)
    f1_tomek.append(f1_score(y_test, DecisionTreeClassifier(criterion='entropy', max_depth=15, min_samples_leaf=5, random_state=42).fit(Xt, yt).predict(X_test), average='macro'))
    print(f"  fold {fold_num+1}: tomek done")

    smote_st = SMOTE(sampling_strategy=target_capped, random_state=42, k_neighbors=5)
    Xst, yst = SMOTETomek(smote=smote_st, random_state=42, n_jobs=-1).fit_resample(X_train, y_train)
    f1_smote_tomek.append(f1_score(y_test, DecisionTreeClassifier(criterion='entropy', max_depth=15, min_samples_leaf=5, random_state=42).fit(Xst, yst).predict(X_test), average='macro'))
    print(f"fold {fold_num+1}/5 complete\n")

print(f"SMOTE, flat target:       {np.mean(f1_flat):.4f} (± {np.std(f1_flat):.4f})")
print(f"SMOTE, ratio-capped 5x:   {np.mean(f1_smote):.4f} (± {np.std(f1_smote):.4f})")
print(f"Random oversampling:      {np.mean(f1_ros):.4f} (± {np.std(f1_ros):.4f})")
print(f"Tomek links alone:        {np.mean(f1_tomek):.4f} (± {np.std(f1_tomek):.4f})")
print(f"SMOTE + Tomek:            {np.mean(f1_smote_tomek):.4f} (± {np.std(f1_smote_tomek):.4f})")

t_stat, p_val = stats.ttest_rel(f1_smote_tomek, f1_smote)
print(f"\nPaired t-test, SMOTE+Tomek vs SMOTE: t={t_stat:.4f}, p={p_val:.4f}")

  fold 1: flat done
  fold 1: smote done
  fold 1: ros done
  fold 1: tomek done
fold 1/5 complete

  fold 2: flat done
  fold 2: smote done
  fold 2: ros done
  fold 2: tomek done
fold 2/5 complete

  fold 3: flat done
  fold 3: smote done
  fold 3: ros done
  fold 3: tomek done
fold 3/5 complete

  fold 4: flat done
  fold 4: smote done
  fold 4: ros done
  fold 4: tomek done
fold 4/5 complete

  fold 5: flat done
  fold 5: smote done
  fold 5: ros done
  fold 5: tomek done
fold 5/5 complete

SMOTE, flat target:       0.5565 (± 0.0047)
SMOTE, ratio-capped 5x:   0.5677 (± 0.0093)
Random oversampling:      0.5755 (± 0.0067)
Tomek links alone:        0.5476 (± 0.0034)
SMOTE + Tomek:            0.5622 (± 0.0090)

Paired t-test, SMOTE+Tomek vs SMOTE: t=-1.5068, p=0.2063


In [6]:
import json
resampling_results = {
    'f1_flat': f1_flat, 'f1_smote': f1_smote, 'f1_ros': f1_ros,
    'f1_tomek': f1_tomek, 'f1_smote_tomek': f1_smote_tomek
}
with open('/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 2/tree_resampling_results.json', 'w') as f:
    json.dump(resampling_results, f)
print("Saved.")

Saved.


In [7]:
# is ROS actually significantly better, or within noise
t_stat2, p_val2 = stats.ttest_rel(f1_ros, f1_smote)
print(f"Paired t-test, ROS vs SMOTE ratio-capped: t={t_stat2:.4f}, p={p_val2:.4f}")

t_stat3, p_val3 = stats.ttest_rel(f1_ros, f1_smote_tomek)
print(f"Paired t-test, ROS vs SMOTE+Tomek: t={t_stat3:.4f}, p={p_val3:.4f}")

Paired t-test, ROS vs SMOTE ratio-capped: t=3.5689, p=0.0234
Paired t-test, ROS vs SMOTE+Tomek: t=3.0529, p=0.0379


Random Oversampling adopted. Best macro-F1 (0.5505), significantly ahead of SMOTE+Tomek (p=0.0142), directionally ahead of plain SMOTE (p=0.1018, likely underpowered with only 5 folds). Also the more defensible choice mechanistically: SMOTE interpolates between neighbours, which on one-hot proto/state/service columns can produce fractional values that are not real categories, while ROS only duplicates real rows. max_depth/min_samples_leaf re-validated on ROS-resampled data since class balance has changed.



# Refined depth and leaf-size search

In [8]:
# re-validate max_depth / min_samples_leaf after ROS
# single representative fold again
X_train_f, X_test_f, y_train_f, y_test_f = fold_data_full[0]
target_capped_f = {cls: min(count * 5, y_train_f.value_counts().max()) for cls, count in y_train_f.value_counts().items()}
X_ros_f, y_ros_f = RandomOverSampler(sampling_strategy=target_capped_f, random_state=42).fit_resample(X_train_f, y_train_f)

depths = [5, 10, 15, 20, 25, None]
leaf_sizes = [1, 5, 20, 50]

grid_results_ros = []
for depth in depths:
    for leaf in leaf_sizes:
        model = DecisionTreeClassifier(criterion='entropy', max_depth=depth,
                                        min_samples_leaf=leaf, random_state=42)
        model.fit(X_ros_f, y_ros_f)
        macro_f1 = f1_score(y_test_f, model.predict(X_test_f), average='macro')
        grid_results_ros.append({'max_depth': depth, 'min_samples_leaf': leaf, 'macro_f1': macro_f1})
        print(f"max_depth={str(depth):5s} min_samples_leaf={leaf:3d}  macro-F1={macro_f1:.4f}")

best_ros = max(grid_results_ros, key=lambda r: r["macro_f1"])
print("\nBest after ROS:", best_ros)

max_depth=5     min_samples_leaf=  1  macro-F1=0.3614
max_depth=5     min_samples_leaf=  5  macro-F1=0.3614
max_depth=5     min_samples_leaf= 20  macro-F1=0.3616
max_depth=5     min_samples_leaf= 50  macro-F1=0.3613
max_depth=10    min_samples_leaf=  1  macro-F1=0.5565
max_depth=10    min_samples_leaf=  5  macro-F1=0.5563
max_depth=10    min_samples_leaf= 20  macro-F1=0.5565
max_depth=10    min_samples_leaf= 50  macro-F1=0.5541
max_depth=15    min_samples_leaf=  1  macro-F1=0.5817
max_depth=15    min_samples_leaf=  5  macro-F1=0.5839
max_depth=15    min_samples_leaf= 20  macro-F1=0.5878
max_depth=15    min_samples_leaf= 50  macro-F1=0.5778
max_depth=20    min_samples_leaf=  1  macro-F1=0.5612
max_depth=20    min_samples_leaf=  5  macro-F1=0.5654
max_depth=20    min_samples_leaf= 20  macro-F1=0.5633
max_depth=20    min_samples_leaf= 50  macro-F1=0.5687
max_depth=25    min_samples_leaf=  1  macro-F1=0.5540
max_depth=25    min_samples_leaf=  5  macro-F1=0.5594
max_depth=25    min_samples_

In [9]:
# confirm entropy vs gini on the final adopted config before the full run
model_entropy = DecisionTreeClassifier(criterion='entropy', max_depth=15, min_samples_leaf=20, random_state=42)
model_entropy.fit(X_ros_f, y_ros_f)
f1_entropy_final = f1_score(y_test_f, model_entropy.predict(X_test_f), average='macro')

model_gini = DecisionTreeClassifier(criterion='gini', max_depth=15, min_samples_leaf=20, random_state=42)
model_gini.fit(X_ros_f, y_ros_f)
f1_gini_final = f1_score(y_test_f, model_gini.predict(X_test_f), average='macro')

print(f"entropy, depth=15, leaf=20: {f1_entropy_final:.4f}")
print(f"gini,    depth=15, leaf=20: {f1_gini_final:.4f}")

entropy, depth=15, leaf=20: 0.5878
gini,    depth=15, leaf=20: 0.5872


Final config confirmed: criterion=entropy, max_depth=15, min_samples_leaf=20, trained on Random Oversampling-resampled data. Entropy still clearly ahead of gini at this configuration (0.5489 vs 0.5234), so the earlier choice holds. later there is a full 5-fold evaluation of this exact configuration.

checkpoint at this stage of tuning (depth/leaf caps, still on the two-feature cluster) - not used once ccp_alpha and the seven-feature cluster get adopted below. kept for the record, not the real final result.

In [10]:
# full 5-fold evaluation of the adopted configuration (entropy, max_depth=15, min_samples_leaf=20, Random Oversampling)
final_f1, final_acc, final_weighted = [], [], []
all_y_test, all_y_pred = [], []

for X_train, X_test, y_train, y_test in fold_data_full:
    target_capped = {cls: min(count * 5, y_train.value_counts().max()) for cls, count in y_train.value_counts().items()}
    Xr, yr = RandomOverSampler(sampling_strategy=target_capped, random_state=42).fit_resample(X_train, y_train)

    model = DecisionTreeClassifier(criterion='entropy', max_depth=15, min_samples_leaf=20, random_state=42)
    model.fit(Xr, yr)
    y_pred = model.predict(X_test)

    final_f1.append(f1_score(y_test, y_pred, average='macro'))
    final_acc.append(accuracy_score(y_test, y_pred))
    final_weighted.append(f1_score(y_test, y_pred, average='weighted'))
    all_y_test.extend(y_test)
    all_y_pred.extend(y_pred)

print(f"Mean macro-F1:    {np.mean(final_f1):.4f} (± {np.std(final_f1):.4f})")
print(f"Mean accuracy:    {np.mean(final_acc):.4f} (± {np.std(final_acc):.4f})")
print(f"Mean weighted-F1: {np.mean(final_weighted):.4f} (± {np.std(final_weighted):.4f})")
print()
print(classification_report(all_y_test, all_y_pred, digits=3))

Mean macro-F1:    0.5687 (± 0.0115)
Mean accuracy:    0.7573 (± 0.0038)
Mean weighted-F1: 0.7749 (± 0.0027)

              precision    recall  f1-score   support

           0      0.982     0.766     0.861     85722
           1      0.688     0.795     0.737      9991
           2      0.167     0.195     0.179      1880
           3      0.290     0.339     0.312      5500
           4      0.828     0.779     0.803     27434
           5      0.124     0.180     0.147      2032
           6      0.475     0.853     0.611     20960
           7      0.526     0.526     0.526       171
           8      0.563     0.696     0.622      1456
           9      0.908     0.877     0.892      7599

    accuracy                          0.757    162745
   macro avg      0.555     0.601     0.569    162745
weighted avg      0.821     0.757     0.775    162745



In [11]:
# Testing some more around the depth=15/leaf=20 winner
# depth=15 and leaf=20 both sat mid-grid, but the spacing was coarse - checking nearby values in case something slightly better was skipped

depths_fine = [12, 13, 14, 15, 16, 17, 18]
leaf_sizes_fine = [10, 15, 20, 25, 30]

grid_results_fine = []
for depth in depths_fine:
    for leaf in leaf_sizes_fine:
        model = DecisionTreeClassifier(criterion='entropy', max_depth=depth,
                                        min_samples_leaf=leaf, random_state=42)
        model.fit(X_ros_f, y_ros_f)
        macro_f1 = f1_score(y_test_f, model.predict(X_test_f), average='macro')
        grid_results_fine.append({'max_depth': depth, 'min_samples_leaf': leaf, 'macro_f1': macro_f1})
        print(f"max_depth={depth:3d} min_samples_leaf={leaf:3d}  macro-F1={macro_f1:.4f}")

best_fine = max(grid_results_fine, key=lambda r: r["macro_f1"])
print("\nBest (fine grid):", best_fine)
print("Previous best (coarse grid): max_depth=15, min_samples_leaf=20, macro_f1=0.5489")

max_depth= 12 min_samples_leaf= 10  macro-F1=0.5851
max_depth= 12 min_samples_leaf= 15  macro-F1=0.5862
max_depth= 12 min_samples_leaf= 20  macro-F1=0.5852
max_depth= 12 min_samples_leaf= 25  macro-F1=0.5833
max_depth= 12 min_samples_leaf= 30  macro-F1=0.5805
max_depth= 13 min_samples_leaf= 10  macro-F1=0.5777
max_depth= 13 min_samples_leaf= 15  macro-F1=0.5822
max_depth= 13 min_samples_leaf= 20  macro-F1=0.5806
max_depth= 13 min_samples_leaf= 25  macro-F1=0.5809
max_depth= 13 min_samples_leaf= 30  macro-F1=0.5794
max_depth= 14 min_samples_leaf= 10  macro-F1=0.5866
max_depth= 14 min_samples_leaf= 15  macro-F1=0.5901
max_depth= 14 min_samples_leaf= 20  macro-F1=0.5897
max_depth= 14 min_samples_leaf= 25  macro-F1=0.5865
max_depth= 14 min_samples_leaf= 30  macro-F1=0.5805
max_depth= 15 min_samples_leaf= 10  macro-F1=0.5802
max_depth= 15 min_samples_leaf= 15  macro-F1=0.5846
max_depth= 15 min_samples_leaf= 20  macro-F1=0.5878
max_depth= 15 min_samples_leaf= 25  macro-F1=0.5828
max_depth= 1

In [12]:
# depth=12 won at the edge of the last grid - check shallower
depths_shallow = [8, 9, 10, 11, 12]
leaf_sizes_shallow = [5, 10, 15, 20, 30]

grid_results_shallow = []
for depth in depths_shallow:
    for leaf in leaf_sizes_shallow:
        model = DecisionTreeClassifier(criterion='entropy', max_depth=depth,
                                        min_samples_leaf=leaf, random_state=42)
        model.fit(X_ros_f, y_ros_f)
        macro_f1 = f1_score(y_test_f, model.predict(X_test_f), average='macro')
        grid_results_shallow.append({'max_depth': depth, 'min_samples_leaf': leaf, 'macro_f1': macro_f1})
        print(f"max_depth={depth:3d} min_samples_leaf={leaf:3d}  macro-F1={macro_f1:.4f}")

best_shallow = max(grid_results_shallow, key=lambda r: r["macro_f1"])
print("\nBest (shallow grid):", best_shallow)
print("Previous best (fine grid): max_depth=12, min_samples_leaf=10, macro_f1=0.5544")

max_depth=  8 min_samples_leaf=  5  macro-F1=0.4963
max_depth=  8 min_samples_leaf= 10  macro-F1=0.4963
max_depth=  8 min_samples_leaf= 15  macro-F1=0.4962
max_depth=  8 min_samples_leaf= 20  macro-F1=0.4962
max_depth=  8 min_samples_leaf= 30  macro-F1=0.4961
max_depth=  9 min_samples_leaf=  5  macro-F1=0.5442
max_depth=  9 min_samples_leaf= 10  macro-F1=0.5443
max_depth=  9 min_samples_leaf= 15  macro-F1=0.5445
max_depth=  9 min_samples_leaf= 20  macro-F1=0.5438
max_depth=  9 min_samples_leaf= 30  macro-F1=0.5436
max_depth= 10 min_samples_leaf=  5  macro-F1=0.5563
max_depth= 10 min_samples_leaf= 10  macro-F1=0.5564
max_depth= 10 min_samples_leaf= 15  macro-F1=0.5567
max_depth= 10 min_samples_leaf= 20  macro-F1=0.5565
max_depth= 10 min_samples_leaf= 30  macro-F1=0.5560
max_depth= 11 min_samples_leaf=  5  macro-F1=0.5837
max_depth= 11 min_samples_leaf= 10  macro-F1=0.5838
max_depth= 11 min_samples_leaf= 15  macro-F1=0.5833
max_depth= 11 min_samples_leaf= 20  macro-F1=0.5841
max_depth= 1

In [13]:
# leaf=5 won at the edge - check smaller leaf sizes too
# depth range narrowed to where the peak actually was (10-13)

depths_narrow = [10, 11, 12, 13]
leaf_sizes_narrow = [1, 2, 3, 5, 7]

grid_results_narrow = []
for depth in depths_narrow:
    for leaf in leaf_sizes_narrow:
        model = DecisionTreeClassifier(criterion='entropy', max_depth=depth,
                                        min_samples_leaf=leaf, random_state=42)
        model.fit(X_ros_f, y_ros_f)
        macro_f1 = f1_score(y_test_f, model.predict(X_test_f), average='macro')
        grid_results_narrow.append({'max_depth': depth, 'min_samples_leaf': leaf, 'macro_f1': macro_f1})
        print(f"max_depth={depth:3d} min_samples_leaf={leaf:3d}  macro-F1={macro_f1:.4f}")

best_narrow = max(grid_results_narrow, key=lambda r: r["macro_f1"])
print("\nBest (narrow grid):", best_narrow)
print("Previous best (shallow grid): max_depth=11, min_samples_leaf=5, macro_f1=0.5561")

max_depth= 10 min_samples_leaf=  1  macro-F1=0.5565
max_depth= 10 min_samples_leaf=  2  macro-F1=0.5563
max_depth= 10 min_samples_leaf=  3  macro-F1=0.5564
max_depth= 10 min_samples_leaf=  5  macro-F1=0.5563
max_depth= 10 min_samples_leaf=  7  macro-F1=0.5565
max_depth= 11 min_samples_leaf=  1  macro-F1=0.5825
max_depth= 11 min_samples_leaf=  2  macro-F1=0.5839
max_depth= 11 min_samples_leaf=  3  macro-F1=0.5837
max_depth= 11 min_samples_leaf=  5  macro-F1=0.5837
max_depth= 11 min_samples_leaf=  7  macro-F1=0.5838
max_depth= 12 min_samples_leaf=  1  macro-F1=0.5839
max_depth= 12 min_samples_leaf=  2  macro-F1=0.5837
max_depth= 12 min_samples_leaf=  3  macro-F1=0.5849
max_depth= 12 min_samples_leaf=  5  macro-F1=0.5834
max_depth= 12 min_samples_leaf=  7  macro-F1=0.5834
max_depth= 13 min_samples_leaf=  1  macro-F1=0.5783
max_depth= 13 min_samples_leaf=  2  macro-F1=0.5789
max_depth= 13 min_samples_leaf=  3  macro-F1=0.5793
max_depth= 13 min_samples_leaf=  5  macro-F1=0.5778
max_depth= 1

# Cost-complexity pruning tested as an alternative to depth/leaf caps

In [14]:
# cost-complexity pruning as an alternative to depth/leaf caps  rather than capping depth/leaf size

model_full = DecisionTreeClassifier(criterion='entropy', random_state=42)
path = model_full.cost_complexity_pruning_path(X_ros_f, y_ros_f)
ccp_alphas = path.ccp_alphas[:-1]  # drop the last one, it prunes to a single node

# too many alpha values to test every one so reduce to 30
idx = np.linspace(0, len(ccp_alphas) - 1, 30).astype(int)
candidate_alphas = ccp_alphas[idx]

ccp_results = []
for alpha in candidate_alphas:
    model = DecisionTreeClassifier(criterion='entropy', random_state=42, ccp_alpha=alpha)
    model.fit(X_ros_f, y_ros_f)
    macro_f1 = f1_score(y_test_f, model.predict(X_test_f), average='macro')
    ccp_results.append({'ccp_alpha': alpha, 'macro_f1': macro_f1, 'n_leaves': model.get_n_leaves()})
    print(f"ccp_alpha={alpha:.6f}  leaves={model.get_n_leaves():4d}  macro-F1={macro_f1:.4f}")

best_ccp = max(ccp_results, key=lambda r: r["macro_f1"])
print("\nBest ccp_alpha:", best_ccp)
print("Depth/leaf-cap winner: max_depth=11, min_samples_leaf=5, macro_f1=0.5561")

ccp_alpha=0.000000  leaves=13958  macro-F1=0.5487
ccp_alpha=0.000005  leaves=13539  macro-F1=0.5500
ccp_alpha=0.000008  leaves=12959  macro-F1=0.5510
ccp_alpha=0.000010  leaves=12415  macro-F1=0.5509
ccp_alpha=0.000011  leaves=11946  macro-F1=0.5509
ccp_alpha=0.000013  leaves=11444  macro-F1=0.5510
ccp_alpha=0.000015  leaves=10951  macro-F1=0.5538
ccp_alpha=0.000016  leaves=10545  macro-F1=0.5537
ccp_alpha=0.000017  leaves=10058  macro-F1=0.5536
ccp_alpha=0.000019  leaves=9614  macro-F1=0.5544
ccp_alpha=0.000020  leaves=9123  macro-F1=0.5553
ccp_alpha=0.000022  leaves=8631  macro-F1=0.5556
ccp_alpha=0.000024  leaves=8069  macro-F1=0.5547
ccp_alpha=0.000026  leaves=7534  macro-F1=0.5588
ccp_alpha=0.000028  leaves=7029  macro-F1=0.5593
ccp_alpha=0.000030  leaves=6516  macro-F1=0.5622
ccp_alpha=0.000032  leaves=6029  macro-F1=0.5605
ccp_alpha=0.000035  leaves=5501  macro-F1=0.5626
ccp_alpha=0.000039  leaves=4992  macro-F1=0.5655
ccp_alpha=0.000042  leaves=4507  macro-F1=0.5706
ccp_alpha=0

In [15]:
# refine ccp_alpha around the 0.000109-0.000180 peak
candidate_alphas_fine = np.linspace(0.0001, 0.00019, 15)

ccp_results_fine = []
for alpha in candidate_alphas_fine:
    model = DecisionTreeClassifier(criterion='entropy', random_state=42, ccp_alpha=alpha)
    model.fit(X_ros_f, y_ros_f)
    macro_f1 = f1_score(y_test_f, model.predict(X_test_f), average='macro')
    ccp_results_fine.append({'ccp_alpha': alpha, 'macro_f1': macro_f1, 'n_leaves': model.get_n_leaves()})
    print(f"ccp_alpha={alpha:.6f}  leaves={model.get_n_leaves():4d}  macro-F1={macro_f1:.4f}")

best_ccp_fine = max(ccp_results_fine, key=lambda r: r["macro_f1"])
print("\nBest ccp_alpha (fine):", best_ccp_fine)

ccp_alpha=0.000100  leaves=1394  macro-F1=0.6011
ccp_alpha=0.000106  leaves=1279  macro-F1=0.6015
ccp_alpha=0.000113  leaves=1158  macro-F1=0.6050
ccp_alpha=0.000119  leaves=1062  macro-F1=0.6065
ccp_alpha=0.000126  leaves= 984  macro-F1=0.6096
ccp_alpha=0.000132  leaves= 921  macro-F1=0.6137
ccp_alpha=0.000139  leaves= 882  macro-F1=0.6124
ccp_alpha=0.000145  leaves= 837  macro-F1=0.6092
ccp_alpha=0.000151  leaves= 793  macro-F1=0.6081
ccp_alpha=0.000158  leaves= 748  macro-F1=0.6073
ccp_alpha=0.000164  leaves= 691  macro-F1=0.6050
ccp_alpha=0.000171  leaves= 654  macro-F1=0.6046
ccp_alpha=0.000177  leaves= 629  macro-F1=0.6034
ccp_alpha=0.000184  leaves= 589  macro-F1=0.6046
ccp_alpha=0.000190  leaves= 567  macro-F1=0.6025

Best ccp_alpha (fine): {'ccp_alpha': np.float64(0.00013214285714285715), 'macro_f1': 0.613715767276742, 'n_leaves': np.int64(921)}


# Classd weighting as an alternative to resampling

In [16]:
# class_weight=balanced as an alternative to ROS
# no resampling - just reweighting the impurity calculation but needs its own pruning path since it is fit on the unresampled training fold
model_full_cw = DecisionTreeClassifier(criterion='entropy', class_weight='balanced', random_state=42)
path_cw = model_full_cw.cost_complexity_pruning_path(X_train_f, y_train_f)
ccp_alphas_cw = path_cw.ccp_alphas[:-1]
idx_cw = np.linspace(0, len(ccp_alphas_cw) - 1, 30).astype(int)
candidate_alphas_cw = ccp_alphas_cw[idx_cw]

ccp_results_cw = []
for alpha in candidate_alphas_cw:
    model = DecisionTreeClassifier(criterion='entropy', class_weight='balanced', random_state=42, ccp_alpha=alpha)
    model.fit(X_train_f, y_train_f)
    macro_f1 = f1_score(y_test_f, model.predict(X_test_f), average='macro')
    ccp_results_cw.append({'ccp_alpha': alpha, 'macro_f1': macro_f1, 'n_leaves': model.get_n_leaves()})
    print(f"ccp_alpha={alpha:.6f}  leaves={model.get_n_leaves():4d}  macro-F1={macro_f1:.4f}")

best_cw = max(ccp_results_cw, key=lambda r: r["macro_f1"])
print("\nBest (class_weight=balanced, no resampling):", best_cw)
print("ROS + ccp_alpha winner: ccp_alpha=0.000158, macro_f1=0.5845")

ccp_alpha=0.000000  leaves=15356  macro-F1=0.5611
ccp_alpha=0.000000  leaves=14998  macro-F1=0.5611
ccp_alpha=0.000000  leaves=14675  macro-F1=0.5611
ccp_alpha=0.000000  leaves=14345  macro-F1=0.5611
ccp_alpha=0.000000  leaves=14013  macro-F1=0.5611
ccp_alpha=0.000003  leaves=13501  macro-F1=0.5616
ccp_alpha=0.000004  leaves=12725  macro-F1=0.5625
ccp_alpha=0.000005  leaves=12121  macro-F1=0.5628
ccp_alpha=0.000007  leaves=11487  macro-F1=0.5635
ccp_alpha=0.000007  leaves=10928  macro-F1=0.5636
ccp_alpha=0.000008  leaves=10407  macro-F1=0.5642
ccp_alpha=0.000010  leaves=9823  macro-F1=0.5650
ccp_alpha=0.000011  leaves=9285  macro-F1=0.5657
ccp_alpha=0.000012  leaves=8695  macro-F1=0.5658
ccp_alpha=0.000014  leaves=8016  macro-F1=0.5670
ccp_alpha=0.000016  leaves=7342  macro-F1=0.5671
ccp_alpha=0.000018  leaves=6741  macro-F1=0.5667
ccp_alpha=0.000021  leaves=6220  macro-F1=0.5673
ccp_alpha=0.000024  leaves=5668  macro-F1=0.5678
ccp_alpha=0.000027  leaves=5143  macro-F1=0.5673
ccp_alpha

In [17]:
# ROS + class_weight=balanced combined
# ROS caps minority classes at 5x, majority classes untouched - testing whether weighting on top of that adds any benefit

model_full_combo = DecisionTreeClassifier(criterion='entropy', class_weight='balanced', random_state=42)
path_combo = model_full_combo.cost_complexity_pruning_path(X_ros_f, y_ros_f)
ccp_alphas_combo = path_combo.ccp_alphas[:-1]
idx_combo = np.linspace(0, len(ccp_alphas_combo) - 1, 30).astype(int)
candidate_alphas_combo = ccp_alphas_combo[idx_combo]

ccp_results_combo = []
for alpha in candidate_alphas_combo:
    model = DecisionTreeClassifier(criterion='entropy', class_weight='balanced', random_state=42, ccp_alpha=alpha)
    model.fit(X_ros_f, y_ros_f)
    macro_f1 = f1_score(y_test_f, model.predict(X_test_f), average='macro')
    ccp_results_combo.append({'ccp_alpha': alpha, 'macro_f1': macro_f1, 'n_leaves': model.get_n_leaves()})
    print(f"ccp_alpha={alpha:.6f}  leaves={model.get_n_leaves():4d}  macro-F1={macro_f1:.4f}")

best_combo = max(ccp_results_combo, key=lambda r: r["macro_f1"])
print("\nBest (ROS + class_weight=balanced):", best_combo)
print("ROS alone + ccp_alpha winner: ccp_alpha=0.000158, macro_f1=0.5845")

ccp_alpha=0.000000  leaves=16040  macro-F1=0.5645
ccp_alpha=0.000000  leaves=15633  macro-F1=0.5645
ccp_alpha=0.000000  leaves=15262  macro-F1=0.5645
ccp_alpha=0.000000  leaves=14908  macro-F1=0.5645
ccp_alpha=0.000000  leaves=14531  macro-F1=0.5645
ccp_alpha=0.000000  leaves=14152  macro-F1=0.5645
ccp_alpha=0.000003  leaves=13498  macro-F1=0.5655
ccp_alpha=0.000005  leaves=12745  macro-F1=0.5659
ccp_alpha=0.000006  leaves=12163  macro-F1=0.5666
ccp_alpha=0.000007  leaves=11564  macro-F1=0.5668
ccp_alpha=0.000008  leaves=11003  macro-F1=0.5681
ccp_alpha=0.000009  leaves=10422  macro-F1=0.5682
ccp_alpha=0.000010  leaves=9872  macro-F1=0.5687
ccp_alpha=0.000011  leaves=9252  macro-F1=0.5694
ccp_alpha=0.000012  leaves=8619  macro-F1=0.5691
ccp_alpha=0.000014  leaves=7914  macro-F1=0.5692
ccp_alpha=0.000016  leaves=7254  macro-F1=0.5703
ccp_alpha=0.000018  leaves=6621  macro-F1=0.5699
ccp_alpha=0.000021  leaves=6014  macro-F1=0.5696
ccp_alpha=0.000024  leaves=5495  macro-F1=0.5681
ccp_alph

# Retention of connection count cluster

rough first pass at this question, single fold - redone properly a couple of cells down, that one is the number actually used.

In [18]:
# cluster-feature question - keep all 7 vs the current 2
# rebuilding from the raw CSV since the 5 dropped features never made it

RAW_PATH = '/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 2/networkTraffic.csv'
df_raw = pd.read_csv(RAW_PATH)
df_cluster_test = df_raw.drop(columns=['id', 'ct_ftp_cmd', 'sloss', 'dloss', 'tcprtt'])

state_mode_ct = df_cluster_test['state'].mode()[0]
df_cluster_test['state'] = df_cluster_test['state'].replace('no', state_mode_ct)
df_cluster_test['is_ftp_login'] = df_cluster_test['is_ftp_login'].replace({2: 1, 4: 1})

# no cluster drop this time - all 7 connection-count features stay in
df_cluster_test = df_cluster_test.drop_duplicates()
print("Shape with all 7 cluster features:", df_cluster_test.shape)

X_ct = df_cluster_test.drop(columns=['attack_cat'])
y_ct = df_cluster_test['attack_cat']
skf_ct = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_idx_ct, test_idx_ct = next(skf_ct.split(X_ct, y_ct))
X_train_ct, X_test_ct = X_ct.iloc[train_idx_ct].copy(), X_ct.iloc[test_idx_ct].copy()
y_train_ct, y_test_ct = y_ct.iloc[train_idx_ct], y_ct.iloc[test_idx_ct]

proto_counts_ct = X_train_ct['proto'].value_counts()
threshold_ct = 0.01 * len(X_train_ct)
keep_protos_ct = proto_counts_ct[proto_counts_ct >= threshold_ct].index.tolist()
X_train_ct['proto'] = X_train_ct['proto'].where(X_train_ct['proto'].isin(keep_protos_ct), 'other')
X_test_ct['proto'] = X_test_ct['proto'].where(X_test_ct['proto'].isin(keep_protos_ct), 'other')

X_train_ct_enc = pd.get_dummies(X_train_ct, columns=nominal_cols)
X_test_ct_enc = pd.get_dummies(X_test_ct, columns=nominal_cols)
X_test_ct_enc = X_test_ct_enc.reindex(columns=X_train_ct_enc.columns, fill_value=0)

target_capped_ct = {cls: min(count * 5, y_train_ct.value_counts().max()) for cls, count in y_train_ct.value_counts().items()}
X_ros_ct, y_ros_ct = RandomOverSampler(sampling_strategy=target_capped_ct, random_state=42).fit_resample(X_train_ct_enc, y_train_ct)

model_ct = DecisionTreeClassifier(criterion='entropy', random_state=42, ccp_alpha=0.000158)
model_ct.fit(X_ros_ct, y_ros_ct)
macro_f1_ct = f1_score(y_test_ct, model_ct.predict(X_test_ct_enc), average='macro')
print(f"All 7 cluster features, same config: macro-F1={macro_f1_ct:.4f}")
print("Current 2-feature version: macro_f1=0.5845")

Shape with all 7 cluster features: (162745, 39)
All 7 cluster features, same config: macro-F1=0.6073
Current 2-feature version: macro_f1=0.5845


In [19]:
# confirm ROS still wins on the 7-feature dataset
# single representative fold, ROS vs current best
X_train_f, X_test_f, y_train_f, y_test_f = fold_data_full[0]

target_capped_f = {cls: min(count * 5, y_train_f.value_counts().max()) for cls, count in y_train_f.value_counts().items()}

X_ros_f, y_ros_f = RandomOverSampler(sampling_strategy=target_capped_f, random_state=42).fit_resample(X_train_f, y_train_f)
model_ros = DecisionTreeClassifier(criterion='entropy', random_state=42).fit(X_ros_f, y_ros_f)
f1_ros_check = f1_score(y_test_f, model_ros.predict(X_test_f), average='macro')

X_smote_f, y_smote_f = SMOTE(sampling_strategy=target_capped_f, random_state=42, k_neighbors=5).fit_resample(X_train_f, y_train_f)
model_smote = DecisionTreeClassifier(criterion='entropy', random_state=42).fit(X_smote_f, y_smote_f)
f1_smote_check = f1_score(y_test_f, model_smote.predict(X_test_f), average='macro')

print(f"ROS:                   {f1_ros_check:.4f}")
print(f"SMOTE ratio-capped 5x: {f1_smote_check:.4f}")

ROS:                   0.5487
SMOTE ratio-capped 5x: 0.5370


In [20]:
# self-contained cluster-feature ablation - builds both the 2-feature and 7-feature representations fresh from the raw CSV in this one cell

RAW_PATH = '/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 2/networkTraffic.csv'
df_raw_check = pd.read_csv(RAW_PATH)
df_base = df_raw_check.drop(columns=['id', 'ct_ftp_cmd', 'sloss', 'dloss', 'tcprtt'])
state_mode_chk = df_base['state'].mode()[0]
df_base['state'] = df_base['state'].replace('no', state_mode_chk)
df_base['is_ftp_login'] = df_base['is_ftp_login'].replace({2: 1, 4: 1})

cluster_cols = ['ct_dst_src_ltm', 'ct_src_dport_ltm', 'ct_dst_ltm', 'ct_src_ltm', 'ct_dst_sport_ltm']

def build_and_eval(df_variant, label):
    df_v = df_variant.drop_duplicates()
    Xv = df_v.drop(columns=['attack_cat'])
    yv = df_v['attack_cat']
    skf_v = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    train_idx, test_idx = next(skf_v.split(Xv, yv))
    X_train_v, X_test_v = Xv.iloc[train_idx].copy(), Xv.iloc[test_idx].copy()
    y_train_v, y_test_v = yv.iloc[train_idx], yv.iloc[test_idx]

    proto_counts_v = X_train_v['proto'].value_counts()
    threshold_v = 0.01 * len(X_train_v)
    keep_protos_v = proto_counts_v[proto_counts_v >= threshold_v].index.tolist()
    X_train_v['proto'] = X_train_v['proto'].where(X_train_v['proto'].isin(keep_protos_v), 'other')
    X_test_v['proto'] = X_test_v['proto'].where(X_test_v['proto'].isin(keep_protos_v), 'other')

    X_train_v_enc = pd.get_dummies(X_train_v, columns=nominal_cols)
    X_test_v_enc = pd.get_dummies(X_test_v, columns=nominal_cols)
    X_test_v_enc = X_test_v_enc.reindex(columns=X_train_v_enc.columns, fill_value=0)

    target_v = {cls: min(count * 5, y_train_v.value_counts().max()) for cls, count in y_train_v.value_counts().items()}
    X_ros_v, y_ros_v = RandomOverSampler(sampling_strategy=target_v, random_state=42).fit_resample(X_train_v_enc, y_train_v)

    model_v = DecisionTreeClassifier(criterion='entropy', random_state=42, ccp_alpha=0.000129)
    model_v.fit(X_ros_v, y_ros_v)
    f1_v = f1_score(y_test_v, model_v.predict(X_test_v_enc), average='macro')
    print(f"{label}: shape={df_v.shape}, macro-F1={f1_v:.4f}")
    return f1_v

f1_two = build_and_eval(df_base.drop(columns=cluster_cols), 'Two-feature cluster (ct_srv_src, ct_srv_dst only)')
f1_seven = build_and_eval(df_base.copy(), 'Seven-feature cluster (all retained)')
print(f"\nDifference: {f1_seven - f1_two:.4f}")

Two-feature cluster (ct_srv_src, ct_srv_dst only): shape=(150243, 34), macro-F1=0.5817
Seven-feature cluster (all retained): shape=(162745, 39), macro-F1=0.6130

Difference: 0.0313


# Final configuration and oversampling ratio

In [21]:
# re-tune ccp_alpha on the 7-feature dataset

model_full_new = DecisionTreeClassifier(criterion='entropy', random_state=42)
path_new = model_full_new.cost_complexity_pruning_path(X_ros_f, y_ros_f)
ccp_alphas_new = path_new.ccp_alphas[:-1]
idx_new = np.linspace(0, len(ccp_alphas_new) - 1, 30).astype(int)
candidate_alphas_new = ccp_alphas_new[idx_new]

ccp_results_new = []
for alpha in candidate_alphas_new:
    model = DecisionTreeClassifier(criterion='entropy', random_state=42, ccp_alpha=alpha)
    model.fit(X_ros_f, y_ros_f)
    macro_f1 = f1_score(y_test_f, model.predict(X_test_f), average='macro')
    ccp_results_new.append({'ccp_alpha': alpha, 'macro_f1': macro_f1, 'n_leaves': model.get_n_leaves()})
    print(f"ccp_alpha={alpha:.6f}  leaves={model.get_n_leaves():4d}  macro-F1={macro_f1:.4f}")

best_new = max(ccp_results_new, key=lambda r: r["macro_f1"])
print("\nBest ccp_alpha (coarse, 7-feature data):", best_new)

ccp_alpha=0.000000  leaves=13958  macro-F1=0.5487
ccp_alpha=0.000005  leaves=13539  macro-F1=0.5500
ccp_alpha=0.000008  leaves=12959  macro-F1=0.5510
ccp_alpha=0.000010  leaves=12415  macro-F1=0.5509
ccp_alpha=0.000011  leaves=11946  macro-F1=0.5509
ccp_alpha=0.000013  leaves=11444  macro-F1=0.5510
ccp_alpha=0.000015  leaves=10951  macro-F1=0.5538
ccp_alpha=0.000016  leaves=10545  macro-F1=0.5537
ccp_alpha=0.000017  leaves=10058  macro-F1=0.5536
ccp_alpha=0.000019  leaves=9614  macro-F1=0.5544
ccp_alpha=0.000020  leaves=9123  macro-F1=0.5553
ccp_alpha=0.000022  leaves=8631  macro-F1=0.5556
ccp_alpha=0.000024  leaves=8069  macro-F1=0.5547
ccp_alpha=0.000026  leaves=7534  macro-F1=0.5588
ccp_alpha=0.000028  leaves=7029  macro-F1=0.5593
ccp_alpha=0.000030  leaves=6516  macro-F1=0.5622
ccp_alpha=0.000032  leaves=6029  macro-F1=0.5605
ccp_alpha=0.000035  leaves=5501  macro-F1=0.5626
ccp_alpha=0.000039  leaves=4992  macro-F1=0.5655
ccp_alpha=0.000042  leaves=4507  macro-F1=0.5706
ccp_alpha=0

In [22]:
# refine ccp_alpha around the 0.000111-0.000270 peak
candidate_alphas_new_fine = np.linspace(0.0001, 0.0002, 15)

ccp_results_new_fine = []
for alpha in candidate_alphas_new_fine:
    model = DecisionTreeClassifier(criterion='entropy', random_state=42, ccp_alpha=alpha)
    model.fit(X_ros_f, y_ros_f)
    macro_f1 = f1_score(y_test_f, model.predict(X_test_f), average='macro')
    ccp_results_new_fine.append({'ccp_alpha': alpha, 'macro_f1': macro_f1, 'n_leaves': model.get_n_leaves()})
    print(f"ccp_alpha={alpha:.6f}  leaves={model.get_n_leaves():4d}  macro-F1={macro_f1:.4f}")

best_new_fine = max(ccp_results_new_fine, key=lambda r: r["macro_f1"])
print("\nBest ccp_alpha (fine, 7-feature data):", best_new_fine)

ccp_alpha=0.000100  leaves=1394  macro-F1=0.6011
ccp_alpha=0.000107  leaves=1258  macro-F1=0.6053
ccp_alpha=0.000114  leaves=1132  macro-F1=0.6049
ccp_alpha=0.000121  leaves=1033  macro-F1=0.6085
ccp_alpha=0.000129  leaves= 951  macro-F1=0.6132
ccp_alpha=0.000136  leaves= 896  macro-F1=0.6125
ccp_alpha=0.000143  leaves= 852  macro-F1=0.6098
ccp_alpha=0.000150  leaves= 808  macro-F1=0.6081
ccp_alpha=0.000157  leaves= 757  macro-F1=0.6074
ccp_alpha=0.000164  leaves= 691  macro-F1=0.6050
ccp_alpha=0.000171  leaves= 649  macro-F1=0.6043
ccp_alpha=0.000179  leaves= 616  macro-F1=0.6034
ccp_alpha=0.000186  leaves= 578  macro-F1=0.6044
ccp_alpha=0.000193  leaves= 556  macro-F1=0.6019
ccp_alpha=0.000200  leaves= 534  macro-F1=0.6016

Best ccp_alpha (fine, 7-feature data): {'ccp_alpha': np.float64(0.00012857142857142858), 'macro_f1': 0.613197403116294, 'n_leaves': np.int64(951)}


In [23]:
# confirm entropy vs gini on the final 7-feature config
model_entropy_final = DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.000129, random_state=42)
model_entropy_final.fit(X_ros_f, y_ros_f)
f1_entropy_final2 = f1_score(y_test_f, model_entropy_final.predict(X_test_f), average='macro')

model_gini_final = DecisionTreeClassifier(criterion='gini', ccp_alpha=0.000129, random_state=42)
model_gini_final.fit(X_ros_f, y_ros_f)
f1_gini_final2 = f1_score(y_test_f, model_gini_final.predict(X_test_f), average='macro')

print(f"entropy, ccp_alpha=0.000129: {f1_entropy_final2:.4f}")
print(f"gini,    ccp_alpha=0.000129: {f1_gini_final2:.4f}")

entropy, ccp_alpha=0.000129: 0.6130
gini,    ccp_alpha=0.000129: 0.6003


In [24]:
#  was the 5x ROS cap actually the right choice for trees
# never re-derived for trees - inherited from the kNN pipeline. Testing a range of multipliers, ccp_alpha held fixed at the current best for now
multipliers = [2, 3, 5, 7, 10]
ratio_results = []

for mult in multipliers:
    target_mult = {cls: min(count * mult, y_train_f.value_counts().max()) for cls, count in y_train_f.value_counts().items()}
    X_r, y_r = RandomOverSampler(sampling_strategy=target_mult, random_state=42).fit_resample(X_train_f, y_train_f)
    model = DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.000129, random_state=42)
    model.fit(X_r, y_r)
    macro_f1 = f1_score(y_test_f, model.predict(X_test_f), average='macro')
    ratio_results.append({'multiplier': mult, 'macro_f1': macro_f1})
    print(f"multiplier={mult:3d}x  macro-F1={macro_f1:.4f}")

# also testing full balance - every class oversampled to match the majority exactly
target_full = {cls: y_train_f.value_counts().max() for cls in y_train_f.value_counts().index}
X_full, y_full = RandomOverSampler(sampling_strategy=target_full, random_state=42).fit_resample(X_train_f, y_train_f)
model_full_bal = DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.000129, random_state=42)
model_full_bal.fit(X_full, y_full)
macro_f1_full = f1_score(y_test_f, model_full_bal.predict(X_test_f), average='macro')
print(f"full balance  macro-F1={macro_f1_full:.4f}")

best_ratio = max(ratio_results, key=lambda r: r["macro_f1"])
print("\nBest multiplier:", best_ratio)
print("Current (5x) for comparison:", [r for r in ratio_results if r["multiplier"]==5])

multiplier=  2x  macro-F1=0.5943
multiplier=  3x  macro-F1=0.5921
multiplier=  5x  macro-F1=0.6130
multiplier=  7x  macro-F1=0.6045
multiplier= 10x  macro-F1=0.5972
full balance  macro-F1=0.5984

Best multiplier: {'multiplier': 5, 'macro_f1': 0.6129653271118547}
Current (5x) for comparison: [{'multiplier': 5, 'macro_f1': 0.6129653271118547}]


In [25]:
# full resampling comparison
# same 5-strategy, 5-fold comparison as before, now on the final feature set
from scipy import stats
f1_flat, f1_smote, f1_ros, f1_tomek, f1_smote_tomek = [], [], [], [], []

for fold_num, (X_train, X_test, y_train, y_test) in enumerate(fold_data_full):
    target_flat = {cls: max(count, 5000) for cls, count in y_train.value_counts().items()}
    Xs, ys = SMOTE(sampling_strategy=target_flat, random_state=42, k_neighbors=5).fit_resample(X_train, y_train)
    f1_flat.append(f1_score(y_test, DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.000129, random_state=42).fit(Xs, ys).predict(X_test), average='macro'))
    print(f"  fold {fold_num+1}: flat done")

    target_capped = {cls: min(count * 5, y_train.value_counts().max()) for cls, count in y_train.value_counts().items()}
    Xs, ys = SMOTE(sampling_strategy=target_capped, random_state=42, k_neighbors=5).fit_resample(X_train, y_train)
    f1_smote.append(f1_score(y_test, DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.000129, random_state=42).fit(Xs, ys).predict(X_test), average='macro'))
    print(f"  fold {fold_num+1}: smote done")

    Xr, yr = RandomOverSampler(sampling_strategy=target_capped, random_state=42).fit_resample(X_train, y_train)
    f1_ros.append(f1_score(y_test, DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.000129, random_state=42).fit(Xr, yr).predict(X_test), average='macro'))
    print(f"  fold {fold_num+1}: ros done")

    Xt, yt = TomekLinks(n_jobs=-1).fit_resample(X_train, y_train)
    f1_tomek.append(f1_score(y_test, DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.000129, random_state=42).fit(Xt, yt).predict(X_test), average='macro'))
    print(f"  fold {fold_num+1}: tomek done")

    smote_st = SMOTE(sampling_strategy=target_capped, random_state=42, k_neighbors=5)
    Xst, yst = SMOTETomek(smote=smote_st, random_state=42, n_jobs=-1).fit_resample(X_train, y_train)
    f1_smote_tomek.append(f1_score(y_test, DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.000129, random_state=42).fit(Xst, yst).predict(X_test), average='macro'))
    print(f"fold {fold_num+1}/5 complete\n")

print(f"SMOTE, flat target:       {np.mean(f1_flat):.4f} (± {np.std(f1_flat):.4f})")
print(f"SMOTE, ratio-capped 5x:   {np.mean(f1_smote):.4f} (± {np.std(f1_smote):.4f})")
print(f"Random oversampling:      {np.mean(f1_ros):.4f} (± {np.std(f1_ros):.4f})")
print(f"Tomek links alone:        {np.mean(f1_tomek):.4f} (± {np.std(f1_tomek):.4f})")
print(f"SMOTE + Tomek:            {np.mean(f1_smote_tomek):.4f} (± {np.std(f1_smote_tomek):.4f})")

t_stat, p_val = stats.ttest_rel(f1_ros, f1_smote)
print(f"\nPaired t-test, ROS vs SMOTE ratio-capped: t={t_stat:.4f}, p={p_val:.4f}")
t_stat2, p_val2 = stats.ttest_rel(f1_ros, f1_smote_tomek)
print(f"Paired t-test, ROS vs SMOTE+Tomek: t={t_stat2:.4f}, p={p_val2:.4f}")

  fold 1: flat done
  fold 1: smote done
  fold 1: ros done
  fold 1: tomek done
fold 1/5 complete

  fold 2: flat done
  fold 2: smote done
  fold 2: ros done
  fold 2: tomek done
fold 2/5 complete

  fold 3: flat done
  fold 3: smote done
  fold 3: ros done
  fold 3: tomek done
fold 3/5 complete

  fold 4: flat done
  fold 4: smote done
  fold 4: ros done
  fold 4: tomek done
fold 4/5 complete

  fold 5: flat done
  fold 5: smote done
  fold 5: ros done
  fold 5: tomek done
fold 5/5 complete

SMOTE, flat target:       0.5744 (± 0.0046)
SMOTE, ratio-capped 5x:   0.5888 (± 0.0058)
Random oversampling:      0.6006 (± 0.0081)
Tomek links alone:        0.5775 (± 0.0061)
SMOTE + Tomek:            0.5838 (± 0.0078)

Paired t-test, ROS vs SMOTE ratio-capped: t=6.8977, p=0.0023
Paired t-test, ROS vs SMOTE+Tomek: t=8.9017, p=0.0009


# Final model

In [26]:
# full 5-fold evaluation
# configuration - entropy, ccp_alpha=0.000129, Random Oversampling, all 7 connection-count features retained

final_f1, final_acc, final_weighted = [], [], []
all_y_test, all_y_pred = [], []

for X_train, X_test, y_train, y_test in fold_data_full:
    target_capped = {cls: min(count * 5, y_train.value_counts().max()) for cls, count in y_train.value_counts().items()}
    Xr, yr = RandomOverSampler(sampling_strategy=target_capped, random_state=42).fit_resample(X_train, y_train)

    model = DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.000129, random_state=42)
    model.fit(Xr, yr)
    y_pred = model.predict(X_test)

    final_f1.append(f1_score(y_test, y_pred, average='macro'))
    final_acc.append(accuracy_score(y_test, y_pred))
    final_weighted.append(f1_score(y_test, y_pred, average='weighted'))
    all_y_test.extend(y_test)
    all_y_pred.extend(y_pred)

print(f"Mean macro-F1:    {np.mean(final_f1):.4f} (± {np.std(final_f1):.4f})")
print(f"Mean accuracy:    {np.mean(final_acc):.4f} (± {np.std(final_acc):.4f})")
print(f"Mean weighted-F1: {np.mean(final_weighted):.4f} (± {np.std(final_weighted):.4f})")
print()
print(classification_report(all_y_test, all_y_pred, digits=3))

Mean macro-F1:    0.6006 (± 0.0081)
Mean accuracy:    0.7723 (± 0.0038)
Mean weighted-F1: 0.7890 (± 0.0035)

              precision    recall  f1-score   support

           0      0.980     0.787     0.873     85722
           1      0.718     0.799     0.757      9991
           2      0.220     0.236     0.228      1880
           3      0.314     0.381     0.344      5500
           4      0.833     0.788     0.810     27434
           5      0.166     0.317     0.218      2032
           6      0.505     0.838     0.630     20960
           7      0.576     0.690     0.628       171
           8      0.548     0.733     0.627      1456
           9      0.917     0.879     0.898      7599

    accuracy                          0.772    162745
   macro avg      0.578     0.645     0.601    162745
weighted avg      0.829     0.772     0.789    162745



In [27]:
# feature importance from the final adopted configuration
# single representative fold
model_final_importance = DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.000129, random_state=42)
model_final_importance.fit(X_ros_f, y_ros_f)

importances = pd.Series(model_final_importance.feature_importances_, index=X_ros_f.columns)
top_15 = importances.sort_values(ascending=False).head(15)
print(top_15)

sttl              0.185548
sbytes            0.158869
service_dns       0.130840
ct_state_ttl      0.116813
smean             0.097537
ct_srv_dst        0.080229
dbytes            0.048024
dmean             0.023535
ct_dst_src_ltm    0.023058
service_http      0.022328
spkts             0.011418
proto_other       0.011025
sjit              0.010459
proto_udp         0.008635
ct_srv_src        0.007742
dtype: float64


In [28]:
# save per-fold macro-F1 so it can be compared against kNN's results
import json
with open('/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 2/tree_final_f1.json', 'w') as f:
    json.dump(final_f1, f)
print("Saved:", final_f1)

Saved: [0.6129653271118547, 0.5944734995405196, 0.603450665624034, 0.6026063481237302, 0.5894821587975576]


In [29]:
# paired t-test: classification tree vs kNN, per-fold macro-F1
import json
from scipy import stats

with open('/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 2/knn_final_f1.json') as f:
    knn_f1 = json.load(f)
with open('/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 2/tree_final_f1.json') as f:
    tree_f1 = json.load(f)

t_stat, p_val = stats.ttest_rel(tree_f1, knn_f1)
print(f"kNN per-fold macro-F1:  {knn_f1}")
print(f"Tree per-fold macro-F1: {tree_f1}")
print(f"\nPaired t-test, tree vs kNN: t={t_stat:.4f}, p={p_val:.4f}")

kNN per-fold macro-F1:  [0.541480618526235, 0.5278417525872214, 0.5542537303755382, 0.5488823402400633, 0.5403736958837997]
Tree per-fold macro-F1: [0.6129653271118547, 0.5944734995405196, 0.603450665624034, 0.6026063481237302, 0.5894821587975576]

Paired t-test, tree vs kNN: t=12.4978, p=0.0002
